# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 20 10
по ключу b берется 20 а iloc 0 берет первый элемент по порядку это 10
2. Алматы NaN Астана NaN Шымкент 3.0
пандас складывает по одинаковым индексам Шымкент есть в обоих поэтому 3.0 а Алматы и Астана только в одном поэтому NaN
3. (2, 2)
фильтр оставляет две строки где x больше 1 а столбца два поэтому размер 2 на 2
4. Series DataFrame
одни скобки возвращают один столбец как Series а двойные возвращают DataFrame
5. [1, 2] 5
после фильтра старые индексы 1 и 2 остаются а iloc 0 1 берет первую строку и второй столбец там 5

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [3]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [4]:
# 1
print("Размер таблицы:", df.shape)
print("\nТипы столбцов:")
print(df.dtypes)
print("\nИнформация о таблице:")
df.info()
missing_days = len(df) - df["delivery_days"].count()
print("\nПропусков в delivery_days:", missing_days)
for col in ["city", "category", "status", "customer_id"]:
    print(f"Уникальных в {col}: {df[col].nunique()}")
print("Уникальных order_id:", df["order_id"].nunique())
print("Дублей order_id:", len(df) - df["order_id"].nunique())

Размер таблицы: (5463, 8)

Типы столбцов:
order_id           int64
order_date           str
customer_id          str
city                 str
category             str
amount           float64
delivery_days    float64
status               str
dtype: object

Информация о таблице:
<class 'pandas.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   str    
 2   customer_id    5463 non-null   str    
 3   city           5463 non-null   str    
 4   category       5463 non-null   str    
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 341.6 KB

Пропусков в delivery_days: 160
Уникальных в city: 14
Уникальных в category: 5
Уникальных в status: 2
Уникальных в customer_id

в таблице 5463 строки и 8 столбцов.
в delivery_days 160 пропусков в city 14 разных написаний вместо 6 городов категорий 5 статусов 2 клиентов 2003.
для отчета по выручке мешают три вещи.
первая это разные написания одного города из за этого легко потерять часть заказов.
вторая это пробелы в числах в amount без очистки они читались как текст.
третья это 180 дублей и тестовый клиент TEST с огромными суммами они сильно завышают выручку.

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [5]:
# 2
clean_amount = df.loc[paid & not_test, "amount"]
mean_val = clean_amount.mean()
median_val = clean_amount.median()
q1 = clean_amount.quantile(0.25)
q3 = clean_amount.quantile(0.75)

print(f"Среднее: {mean_val:.2f}")
print(f"Медиана: {median_val:.2f}")
print(f"25 перцентиль: {q1:.2f}")
print(f"75 перцентиль: {q3:.2f}")

Среднее: 14672.39
Медиана: 10300.00
25 перцентиль: 5880.00
75 перцентиль: 18690.00


по оплаченным заказам без TEST среднее 14672.39 тенге медиана 10300 тенге 25 перцентиль 5880 и 75 перцентиль 18690.
обычный заказ примерно от 5880 до 18690 тенге.
менеджеру лучше показывать медиану 10300 потому что дорогая электроника тянет среднее вверх и кажется что чек выше чем на самом деле.

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [6]:
# 3
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]
cancel = df["status"] == "отменён"

all_cancel = cancel.mean() * 100
sh_cancel = cancel[df["city"].isin(SH)].mean() * 100
al_cancel = cancel[df["city"].isin(AL)].mean() * 100

print(f"Доля отмен по всем заказам: {all_cancel:.2f}%")
print(f"Доля отмен в Шымкенте: {sh_cancel:.2f}%")
print(f"Доля отмен в Алматы: {al_cancel:.2f}%")
print(f"Выше среднего в {sh_cancel / all_cancel:.2f} раза")
print(f"Выше Алматы в {sh_cancel / al_cancel:.2f} раза")

Доля отмен по всем заказам: 10.23%
Доля отмен в Шымкенте: 20.40%
Доля отмен в Алматы: 7.86%
Выше среднего в 1.99 раза
Выше Алматы в 2.59 раза


по всем заказам отменяют 10.23 процента в Алматы 7.86 а в Шымкенте 20.40.
в Шымкенте отмены почти в 2 раза выше среднего и в 2.59 раза выше чем в Алматы.
с доставкой в Шымкенте явно проблема и с ней надо разбираться в первую очередь.

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [7]:
# 4
sh_mask = df["city"].isin(SH)
al_mask = df["city"].isin(AL)

sh_sep = df.loc[paid & not_test & sh_mask & sep, "amount"].sum()
sh_oct = df.loc[paid & not_test & sh_mask & octo, "amount"].sum()
sh_chg = (sh_oct - sh_sep) / sh_sep * 100

al_sep = df.loc[paid & not_test & al_mask & sep, "amount"].sum()
al_oct = df.loc[paid & not_test & al_mask & octo, "amount"].sum()
al_chg = (al_oct - al_sep) / al_sep * 100

print(f"Шымкент сентябрь {sh_sep:.0f}, октябрь {sh_oct:.0f}, изменение {sh_chg:.2f}%")
print(f"Алматы сентябрь {al_sep:.0f}, октябрь {al_oct:.0f}, изменение {al_chg:.2f}%")
print(f"Падение в Шымкенте сильнее в {sh_chg / al_chg:.2f} раза")

Шымкент сентябрь 4949800, октябрь 4044010, изменение -18.30%
Алматы сентябрь 15523940, октябрь 14643020, изменение -5.67%
Падение в Шымкенте сильнее в 3.22 раза


в Шымкенте выручка упала на 18.30 процента с 4949800 до 4044010 тенге.
в Алматы падение 5.67 процента с 15523940 до 14643020 тенге.
в Шымкенте просадка в 3.22 раза сильнее.
правда менеджеру надо сказать что в сентябре еще есть дубли но даже так главная проблема именно в Шымкенте.

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [8]:
# 5
# первая ошибка нет скобок вокруг условий из за этого код падает
# вторая ошибка взяли только одно написание города поэтому теряются остальные варианты
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print("Отмененных заказов в Шымкенте:", len(cancel_sh))

Отмененных заказов в Шымкенте: 164


на самом деле в Шымкенте 164 отмененных заказа.
в коде от ии было две ошибки.
первая что условия не взяли в скобки и код падал.
вторая что взяли только одно написание города и потеряли 11 отмен вышло 153 вместо 164.
поэтому по городам всегда надо проверять все варианты написания.

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [9]:
# 6
top_clients = df["customer_id"].value_counts()
top_id = top_clients.index[0]
print("Топ 5 клиентов по числу заказов:")
print(top_clients.head(5))

top_orders = df.loc[df["customer_id"] == top_id, ["order_date", "city", "amount", "status"]]
print(f"\nЗаказы клиента {top_id}, всего строк {len(top_orders)}:")
print(top_orders)

top_order_ids = df.loc[df["customer_id"] == top_id, "order_id"]
print(f"\nУникальных order_id у {top_id}: {top_order_ids.nunique()} из {len(top_order_ids)}")
print("Повторяющиеся order_id:")
vc_oid = top_order_ids.value_counts()
print(vc_oid[vc_oid > 1])

Топ 5 клиентов по числу заказов:
customer_id
C01818    11
C01076     9
C01947     9
C01656     9
C00420     9
Name: count, dtype: int64

Заказы клиента C01818, всего строк 11:
            order_date       city   amount   status
57    2026-09-01 15:00    Шымкент   7350.0  оплачен
779   2026-09-09 18:00    Шымкент  45270.0  оплачен
1371  2026-09-15 21:00     Алматы   8890.0  оплачен
1375  2026-09-15 21:00     Алматы   8890.0  оплачен
2195  2026-09-24 18:00     Астана   1490.0  оплачен
2617  2026-09-29 09:00     Астана  22170.0  оплачен
2776  2026-09-30 21:00     Алматы  20180.0  оплачен
3417  2026-10-08 14:00  Караганда   4330.0  оплачен
4279  2026-10-18 08:00     Астана  13600.0  оплачен
4610  2026-10-21 19:00     Алматы  21990.0  оплачен
4778  2026-10-23 18:00     Алматы  11140.0  оплачен

Уникальных order_id у C01818: 10 из 11
Повторяющиеся order_id:
order_id
101504    2
Name: count, dtype: int64


чаще всего в таблице встречается клиент C01818 у него 11 строк.
но уникальных order_id только 10 потому что заказ 101504 от 15 сентября на 8890 тенге записан два раза.
если не убрать такие повторы выручка за сентябрь будет завышена а падение в октябре покажется сильнее чем оно есть.

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [10]:
# 7
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
iqr_outliers = df[paid & not_test & (df["amount"] > upper_bound)]
outliers_share = len(iqr_outliers) / len(clean_amount) * 100

print(f"IQR = {iqr:.2f}, верхняя граница = {upper_bound:.2f}")
print(f"Заказов выше границы: {len(iqr_outliers)}, это {outliers_share:.2f}%")
print("\nКатегории выбросов:")
print(iqr_outliers["category"].value_counts())

IQR = 12810.00, верхняя граница = 37905.00
Заказов выше границы: 320, это 6.53%

Категории выбросов:
category
Электроника    308
Одежда          10
Дом              2
Name: count, dtype: int64


по правилу IQR верхняя граница 37905 тенге.
выше этой суммы 320 заказов это 6.53 процента от оплаченных без TEST.
почти все из категории Электроника 308 из 320.
получилось больше чем 87 в прошлом дз потому что дорогая электроника сильно увеличивает среднее и стандартное отклонение и порог по z оценке улетает вверх примерно до 54575.
квартили к таким суммам спокойнее поэтому граница ниже плюс в дз 4 мы еще убирали дубли.

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [11]:
# 8
cancel_by_city = df.loc[df["status"] == "отменён", "city"].value_counts()
all_by_city = df["city"].value_counts()
cancel_rate = (cancel_by_city / all_by_city).sort_values(ascending=False)

print("Доля отмен по каждому написанию города:")
print((cancel_rate * 100).round(2))

print("\nПроверка по вариантам Шымкента:")
for name in SH:
    print(f"{name} всего {all_by_city[name]}, отмен {cancel_by_city[name]}, доля {cancel_rate[name]*100:.2f}%")

Доля отмен по каждому написанию города:
city
Shymkent     23.08
Шымкент      21.52
алматы       14.06
астана       13.10
Актобе       10.33
Astana       10.00
Караганда     8.92
Астана        8.48
Алматы        7.99
Атырау        7.53
Almaty        5.80
 Алматы       4.55
шымкент       3.70
АЛМАТЫ        3.23
Name: count, dtype: float64

Проверка по вариантам Шымкента:
Шымкент всего 711, отмен 153, доля 21.52%
шымкент всего 54, отмен 2, доля 3.70%
Shymkent всего 39, отмен 9, доля 23.08%


пандас делит две серии по одинаковым названиям в индексе а не по порядку строк.
для Шымкента вышло у Shymkent 23.08 процента это 9 из 39 у Шымкент 21.52 процента это 153 из 711 а у шымкент 3.70 процента это 2 из 54.
по одному варианту написания судить о городе нельзя потому что где то заказов мало и процент скачет.
перед анализом столбец city надо почистить убрать пробелы сделать один регистр и перевести латиницу на кириллицу.